# Tutorial 9-1: Scalable Similarity Search with LSH
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. The Performance Realities of LSH
Locality Sensitive Hashing is an **indexing** strategy. Like a database index, it has an upfront cost (building the index) but makes future searches nearly instantaneous. 

**Objective:**
1. Use **Shingling** (character 3-grams) for robust text matching.
2. Demonstrate LSH **Recall** by searching for a synthetically created near-duplicate.
3. Analyze the **Amortized Cost**: Comparing Indexing vs. Querying time.

In [ ]:
!pip install datasketch matplotlib

import time
import matplotlib.pyplot as plt
from datasketch import MinHash, MinHashLSH
from sklearn.datasets import fetch_20newsgroups

# Load the full training set (~11k documents)
print("Loading full dataset...")
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
data = newsgroups.data
print(f"Loaded {len(data)} documents.")

## 2. Pre-processing: Shingling and Signatures
We use character 3-grams and 256 permutations. Note that this pre-processing step is common to both Brute Force and LSH.

In [ ]:
def get_shingles(text, k=3):
    return set([text[i:i+k] for i in range(len(text)-k+1)])

def get_minhash(shingles, num_perm=256):
    m = MinHash(num_perm=num_perm)
    for s in shingles:
        m.update(s.encode('utf8'))
    return m

print("Generating shingles and signatures for all documents...")
num_permutations = 256
doc_shingles = [get_shingles(doc) for doc in data]
signatures = [get_minhash(s, num_perm=num_permutations) for s in doc_shingles]
print("Done.")

## 3. Creating a Near-Duplicate Query
To ensure we have something to find, we will take an existing document and modify it slightly (deleting/changing a few characters) to create a guaranteed near-duplicate.

In [ ]:
target_threshold = 0.20

# Create a synthetic query from document 100
original_text = data[100]
query_text = original_text[:int(len(original_text)*0.9)] + " xyz abc modification"
query_shingles = get_shingles(query_text)
query_sig = get_minhash(query_shingles, num_perm=num_permutations)

print(f"Original Doc 100 length: {len(original_text)}")
print(f"Synthetic Query length: {len(query_text)}")

## 4. Performance Benchmarking
We now perform the search using both methods. For LSH, we track the time taken to build the index versus the time taken to perform the query.

In [ ]:
# --- METHOD 1: BRUTE FORCE ---
print("Running Brute Force Search...")
start_bf = time.time()
bf_results = []
for i, s_set in enumerate(doc_shingles):
    u_len = len(query_shingles.union(s_set))
    if u_len == 0: continue
    jaccard = len(query_shingles.intersection(s_set)) / u_len
    if jaccard >= target_threshold:
        bf_results.append(i)
time_bf = time.time() - start_bf

# --- METHOD 2: LSH INDEXING ---
print("Building LSH Index...")
start_indexing = time.time()
lsh = MinHashLSH(threshold=target_threshold, num_perm=num_permutations)
for i, sig in enumerate(signatures):
    lsh.insert(i, sig)
time_indexing = time.time() - start_indexing

# --- METHOD 3: LSH QUERY & VERIFY ---
print("Running LSH Search with Verification...")
start_lsh_query = time.time()
candidates = lsh.query(query_sig)
lsh_verified_results = []
for cand_id in candidates:
    s_set = doc_shingles[cand_id]
    jaccard = len(query_shingles.intersection(s_set)) / len(query_shingles.union(s_set))
    if jaccard >= target_threshold:
        lsh_verified_results.append(cand_id)
time_lsh_query = time.time() - start_lsh_query

time_lsh_total = time_indexing + time_lsh_query

print(f"\nBrute Force Time: {time_bf:.5f} sec | Neighbors found: {len(bf_results)}")
print(f"LSH Indexing:     {time_indexing:.5f} sec")
print(f"LSH Query+Verify: {time_lsh_query:.5f} sec | Candidates checked: {len(candidates)}")

# Metrics
recall = len(set(bf_results).intersection(set(lsh_verified_results))) / len(bf_results) if bf_results else 1.0
print(f"Recall: {recall:.2%}")

## 5. Visualizing the Advantage
While the **Total Time** for LSH includes a heavy indexing penalty, the **Query Time** is the real metric for production. If we build the index once and perform 1,000 queries, LSH would be hundreds of times faster than Brute Force.

In [ ]:
plt.figure(figsize=(10, 6))
labels = ['Brute Force', 'LSH (Build Index)', 'LSH (Single Query)']
times = [time_bf, time_indexing, time_lsh_query]
colors = ['salmon', 'lightgray', 'skyblue']

plt.bar(labels, times, color=colors)
plt.ylabel('Time (seconds)')
plt.title(f'Performance Breakdown (N={len(data)})')
plt.yscale('log') # Use log scale to visualize the tiny query time

for i, t in enumerate(times):
    plt.text(i, t, f'{t:.5f}s', ha='center', va='bottom', fontweight='bold')

plt.show()